## 第二章 处理文本数据


- 将文本分割为独立的单词词元和子词词元，然后编码为 LLM 所使用的向量表示


### 2.1 理解词嵌入


- 深度神经网络无法直接处理原始文本，因为文本数据是离散的，无法直接执行数学运算。
- 将数据转化为向量格式的过程称为嵌入 embedding，不同的数据类型如文本，图像，音视频等需要使用不同的嵌入模型。
- embedding 的本质是将离散对象映射到连续向量空间中的点，从而转化为神经网络可以处理的格式。
- RAG，retrieval-augmented generation，是将句子，段落乃至整个文档嵌入的技术。
- 有多种算法和框架来生成词嵌入，早期流行 word2vec。
- 词嵌入的维度 dimension 可以从一维到数千维不等，更高的维度有助于捕获到更细微的关系，同时牺牲计算效率。
- 大语言模型通常会自行生成嵌入，这些嵌入是输入层的一部分，并且在训练中会进行更新。
- 最小的 GPT-2 模型参数为 1.17 亿，嵌入维度为 768，GPT-3 参数为 1750 亿，嵌入维度为 12288。


### 2.2 文本分词


- 将输入文本分割为独立的词元，是生成嵌入向量所必须的预处理步骤。
- 词元即可以是单个的单词，也可以是诸如标点符号之类的特殊字符。


- 采用 Edith Wharton 的短篇小说 The Verdict 为分词文本作为例子。


In [1]:
import urllib.request
url = ("https://raw.githubusercontent.com/rasbt/"
       "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
       "the-verdict.txt")
file_path = "the-verdict.txt"
urllib.request.urlretrieve(url, file_path)

('the-verdict.txt', <http.client.HTTPMessage at 0x1f56b3ece10>)

In [2]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
print("Total number of characters in the text:", len(raw_text))
print(raw_text[:99])

Total number of characters in the text: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


- 将这篇 20479 个字符的短片小说分割为独立的单词和特殊字符，以便于在后续步骤中将其转化为嵌入向量，进而用于 LLM 训练。
- 为了获取词元列表，就需要正确的分割文本。
- 何为正确？如移除空格可以减少运算负担，但是会丢失掉空格在上下文中的作用。


- 下面使用正则来示例如何分割


In [3]:
import re
text = "Hello, world. This,  is a test."
result = re.split(r'(\s)', text)
print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', '', ' ', 'is', ' ', 'a', ' ', 'test.']


将单词与标点符号分离


In [4]:
result = re.split(r'([,.]|\s)', text)
print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


空白字符仍然存在，用下面的方式分离


In [8]:
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [9]:
text = "Hello, world. Is this-- a test?"
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


输入了 Hello, world. Is this-- a test?  
输出了'Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?'


In [10]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed))

4690


In [11]:
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [13]:
preprocessed

['I',
 'HAD',
 'always',
 'thought',
 'Jack',
 'Gisburn',
 'rather',
 'a',
 'cheap',
 'genius',
 '--',
 'though',
 'a',
 'good',
 'fellow',
 'enough',
 '--',
 'so',
 'it',
 'was',
 'no',
 'great',
 'surprise',
 'to',
 'me',
 'to',
 'hear',
 'that',
 ',',
 'in',
 'the',
 'height',
 'of',
 'his',
 'glory',
 ',',
 'he',
 'had',
 'dropped',
 'his',
 'painting',
 ',',
 'married',
 'a',
 'rich',
 'widow',
 ',',
 'and',
 'established',
 'himself',
 'in',
 'a',
 'villa',
 'on',
 'the',
 'Riviera',
 '.',
 '(',
 'Though',
 'I',
 'rather',
 'thought',
 'it',
 'would',
 'have',
 'been',
 'Rome',
 'or',
 'Florence',
 '.',
 ')',
 '"',
 'The',
 'height',
 'of',
 'his',
 'glory',
 '"',
 '--',
 'that',
 'was',
 'what',
 'the',
 'women',
 'called',
 'it',
 '.',
 'I',
 'can',
 'hear',
 'Mrs',
 '.',
 'Gideon',
 'Thwing',
 '--',
 'his',
 'last',
 'Chicago',
 'sitter',
 '--',
 'deploring',
 'his',
 'unaccountable',
 'abdication',
 '.',
 '"',
 'Of',
 'course',
 'it',
 "'",
 's',
 'going',
 'to',
 'send',
 't

### 2.3 将词元转换为词元 ID


- 词元被分割后，还需要生成词元 ID，即 token ID，与之一一映射；
- 为了完成映射，需要构建一张词汇表；
- 首先将训练集中的全部文本分割成独立的词元，
- 然后将这些词元按照字母顺序排列并删除重复词元；
- 然后将唯一的词元聚合到一个词汇表中。


In [18]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

1130


In [19]:
vocab = {}

for integer, token in enumerate(all_words):
    # vocab[token] = integer
    print(integer, token)

0 !
1 "
2 '
3 (
4 )
5 ,
6 --
7 .
8 :
9 ;
10 ?
11 A
12 Ah
13 Among
14 And
15 Are
16 Arrt
17 As
18 At
19 Be
20 Begin
21 Burlington
22 But
23 By
24 Carlo
25 Chicago
26 Claude
27 Come
28 Croft
29 Destroyed
30 Devonshire
31 Don
32 Dubarry
33 Emperors
34 Florence
35 For
36 Gallery
37 Gideon
38 Gisburn
39 Gisburns
40 Grafton
41 Greek
42 Grindle
43 Grindles
44 HAD
45 Had
46 Hang
47 Has
48 He
49 Her
50 Hermia
51 His
52 How
53 I
54 If
55 In
56 It
57 Jack
58 Jove
59 Just
60 Lord
61 Made
62 Miss
63 Money
64 Monte
65 Moon-dancers
66 Mr
67 Mrs
68 My
69 Never
70 No
71 Now
72 Nutley
73 Of
74 Oh
75 On
76 Once
77 Only
78 Or
79 Perhaps
80 Poor
81 Professional
82 Renaissance
83 Rickham
84 Riviera
85 Rome
86 Russian
87 Sevres
88 She
89 Stroud
90 Strouds
91 Suddenly
92 That
93 The
94 Then
95 There
96 They
97 This
98 Those
99 Though
100 Thwing
101 Thwings
102 To
103 Usually
104 Venetian
105 Victor
106 Was
107 We
108 Well
109 What
110 When
111 Why
112 Yes
113 You
114 _
115 a
116 abdication
117 able
118 about


In [20]:
vocab = {token: integer for integer, token in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


- 上面演示了对训练文本分割——去重——构建词汇表的过程；
- 现实中还需要构建一个逆向词汇表，从而将词元 ID 映射回对应的文本；
- 下面演示一个完整的分词器，包括了：
- encode, 通过词汇表将文本映射到整数，以生成词元 ID，
- decode，从整数到字符串的反向映射，将词元 ID 还原回文本，


In [21]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab  # 将词汇表作为类存储，以便于在encoder和decoder方法中访问
        # 创建逆向词汇表，将词元ID映射回原始文本词元
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):  # 将输入文本转换为词元ID列表
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):  # 将词元ID列表转换回原始文本
        text = " ".join([self.int_to_str[i] for i in ids])

        text = re.sub(r'\s+([,.:;?_!"()\'])', r'\1', text)  # 移除标点符号前的空格
        return text

创建一个 SimpleTokenizerV1 类的实例对象，将其应用于 The Verdict 的一段文本中，测试效果


In [22]:
tokenizer = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know,"
        Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [23]:
print(tokenizer.decode(ids))

" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


- 分词器仅能用于训练集中的文本里，超出范围就会报错；
- 如下所示，Hello 没有在小说 The Verdict 中出现；


In [14]:
# text = "Hello, do you like tea?"
# print(tokenizer.encode(text))

### 2.4 引入特殊上下文词元


- 为了处理未知单词，需要引入上下文特殊词元
- 特殊词元可能包括标识未知词元和文档边界词元等
- "\<unk>" "\<endoftext>"


In [24]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token: integer for integer, token in enumerate(all_tokens)}

print(len(vocab.items()))

1132


In [25]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


- 修改分词器到 V2 版本


In [26]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        preprocessed = [item if item in self.str_to_int
                        else "<|unk|>" for item in preprocessed]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?_!"()\'])', r'\1', text)
        return text

In [27]:
text1 = "Hello, do you like tea?"
text2 = "in the sunlit terraces of the palace."
text = "<|endoftext|>".join([text1, text2])
print(text)

Hello, do you like tea?<|endoftext|>in the sunlit terraces of the palace.


In [28]:
tokenizer = SimpleTokenizerV2(vocab)
print(tokenizer.encode(text))

[1131, 5, 355, 1126, 628, 975, 10, 1131, 988, 956, 984, 722, 988, 1131, 7]


In [29]:
print(tokenizer.decode(tokenizer.encode(text)))

<|unk|>, do you like tea? <|unk|> the sunlit terraces of the <|unk|>.


- 常见的其它标示符可能还会包含：
- \[BOS]，序列开始
- \[EOS]，序列结束
- \[PAD]，填充。为了让所有文本具有相同的长度，较短的文本会通过添加\[PAD]来进行填充或扩充；


### 2.5 BPE 分词器


- \!pip install tiktoken
- BPE 分词器用于训练大语言模型，比如 GPT-2 和 GPT-3 等。


In [1]:
from importlib.metadata import version
import tiktoken
print(version("tiktoken"))

0.13.0


In [2]:
tokenizer = tiktoken.get_encoding("gpt2")

text = ("Hello, do you like tea? <|endoftext|> In the sunlit terraces"
        "of someunknownPlace.")
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [3]:
# 然后用decode方法将词元ID转换回文本
strings = tokenizer.decode(integers)
print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


- \<|endoftext|> 词元被分配了一个较大的词元 ID，即 50256；日常用于训练 GPT-2，GPT3 和 ChatGPT 中使用的原始模型的 BPE 分词器的词汇总量是 50257
- BPE 分词器可以正确编码解码未知词，因其将不在预定词汇表的单词分解为更小的子词元甚至字符。
- BPE 通过将频繁出现的字符合并为子词，再将频繁出现的子词合并为单词。


### 2.6 使用滑动窗口进行数据采样


- 生成大语言模型的嵌入向量，就是生成用于训练模型的输入-目标对。
- 大语言模型通过预测文本序列中的下一个单词来进行训练。
- 参考 P31，图 2-12


In [24]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


- 采用滑动窗口 sliding window 的方式从训练数据集中提取上图中所示的输入目标对。


- 输入\-目标对用 x 和 y 变量演示；
- x 用于存储输入的词元，y 用于存储由 x 的每个输入词元右移一个位置所得的目标词元；


In [25]:
enc_sample = enc_text[50:]

context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]
print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


In [26]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(context, "---->", desired)

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [27]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


- 词元转化为嵌入向量前，还有最后一个任务：
- 实现一个高效的数据加载器 data loader，其能够遍历输入的数据集，并将输入和目标以 PyTorch 张量的形式返回
- 我们的目标是返回 2 个张量:
- 一个是包含大语言模型所见的文本输入的输入张量；
- 另一个是包含大语言模型需要预测的目标词元的目标张量；
- 参见 P33 图 2-13，演示方便用单词，实际中都是词元 ID


In [28]:
import torch
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt)
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1:i + max_length + 1]

            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]
        DataLoader

In [29]:
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(dataset,
                            batch_size=batch_size,
                            shuffle=shuffle,
                            drop_last=drop_last,
                            num_workers=num_workers
                            )
    return dataloader

In [30]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


- 变量 first_batch 包含 2 个张量: 第一个存储输入的词元 ID，第二个存储目标词元 ID
- max_length 设置为 4 是为了演示，实际训练中，输入大小通常不小于 256；
- stride 即步幅，决定了批次之间输入的位移量，模拟了滑动窗口方法；


- 较小的批次大小会减少训练过程中的内存占用，但同时会导致在模型更新时产生更多的噪声。
- 批次大小是 LLM 训练似乎后需要仔细权衡的超参数。


In [31]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=4, stride=4,
    shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("Targets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


- 上面的例子将步幅增加到 4 来充分利用数据集，这样不会跳过任何一个单词，同时避免了不同批次之间的数据重叠。
- 过多的重叠可能会增加模型过拟合的风险。


### 2.7 创建词元嵌入


- 为 LLM 训练准备输入文本的最后一步是将词元 ID 转换为嵌入向量
- P 37 图 2-15
- 在初始阶段，必须用随机值初始化这些嵌入权重
- 类 GPT 大语言模型是使用反向传播算法 backpropagation algorithm 训练的深度神经网络，因此需要连续的向量表示或嵌入。


- 我们通过一个实际的例子来演示词元 ID 转换为嵌入向量的工作原理。假设有 4 个 ID 分别为 2，3，5，1 的输入词元：


In [32]:
input_ids = torch.tensor([2, 3, 5, 1])

- 为了简单起见，假设我们有一张仅包含 6 个单词的小型词汇表，并且想要创建维度为 3 的嵌入
- 实际中 BPE 分词器中包含有 50257 个单词的词汇表，而 GPT-3 的嵌入维度是 12288


In [ ]:
vacab_size = 6
output_dim = 3

In [40]:
torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.3035],
        [-0.5880,  0.3486,  0.6603],
        [-0.2196, -0.3792,  0.7671],
        ...,
        [-0.7360, -0.3375,  1.1260],
        [ 0.0562,  0.0324,  0.5457],
        [-0.2692,  2.2521,  0.4200]], requires_grad=True)


In [41]:
print(embedding_layer(torch.tensor([2])))

tensor([[-0.2196, -0.3792,  0.7671]], grad_fn=<EmbeddingBackward0>)


- 从上面的打印结果能看到，词元 ID 为 2 的嵌入向量与嵌入矩阵中的第 4 行完全相同
- 嵌入层实际上执行的是一种查找操作，它根据词元 ID 从嵌入层的权重矩阵中检索出相应的行。
- 本质上，可以将嵌入层方法视为一种更有效的实现独立热编码 one-hot encoding 的方法。


- 上面的例子是将单个词元 ID 转换为三维嵌入向量。接下来，我们将这个方法应用到所有的 4 个输入 ID(torch.tensor([2, 3, 5,1]))上


In [43]:
# Parameter containing:
# tensor([[ 0.3374, -0.1778, -0.3035],
#         [-0.5880,  0.3486,  0.6603],
#         [-0.2196, -0.3792,  0.7671],
#         ...,
#         [-0.7360, -0.3375,  1.1260],
#         [ 0.0562,  0.0324,  0.5457],
#         [-0.2692,  2.2521,  0.4200]], requires_grad=True)

In [ ]:
# input_ids = torch.tensor([2, 3, 5, 1])
print(embedding_layer(input_ids))

tensor([[-0.2196, -0.3792,  0.7671],
        [-1.1925,  0.6984, -1.4097],
        [ 0.2692, -0.0770, -1.0205],
        [-0.5880,  0.3486,  0.6603]], grad_fn=<EmbeddingBackward0>)


- 上面的例子就是对词元 ID 执行了 3 维嵌入向量的演示，接下来的工作是对这些嵌入向量进行细微的调整，以编码词元在文本中的位置信息。


### 2.8 编码单词位置信息


- 如上所示，词元 ID 张量的维度为 8x4，这表明数据批次包含 8 个文本样本，本个样本由 4 个词元组成。
- 现在，使用嵌入层将这些词元 ID 嵌入 256 维的向量中：


In [41]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


- 为了获取 GPT 模型所采用的绝对位置嵌入，只需创建一个维度与 token_embedding_layer 相同的嵌入层即可。


In [42]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


P42
